# Multi-model LLM-as-Judge and Robustness Checks

This notebook extends the LLM-as-judge evaluation with:

- GPT-5 as a judge through the OpenAI API.
- Claude Sonnet 4.6 as a judge through the direct Claude API, or optionally through aixplain.
- Gemini as a judge through aixplain.
- Per-dimension score averaging across enabled judges.
- A GPT-5 perturbation experiment that reduces either the input bundle or generated output by 0.05, 0.10, and 0.15 while keeping the paired side unchanged.

The Brazilian manager output is included explicitly:

- `results/nlg_brazilian_manager/default/pt_br_manager_report_br_manager_2024-01-02.json`
- `results/nlg_brazilian_manager/default/pt_br_manager_report_br_manager_2024-01-02.txt`
- `results/nlg_brazilian_manager/e2e/pt_br_manager_report_br_manager_2024-01-02.json`
- `results/nlg_brazilian_manager/e2e/pt_br_manager_report_br_manager_2024-01-02.txt`


In [ ]:
from pathlib import Path
import statistics as stats
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Financial-D2T-Agent":
    PROJECT_ROOT = PROJECT_ROOT / "Financial-D2T-Agent"

RESULTS_ROOT = PROJECT_ROOT / "results"

CATEGORIES = [
    "nlg_brazilian_manager/e2e",
    "nlg_brazilian_manager/default",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_orchestrator_no_guardrail_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_orchestrator_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/e2e",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/default_old",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/default",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/e2e",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/default_old",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/default",
]

try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-5")
    def count_tokens(text: str) -> int:
        return len(enc.encode(text))
    tokenizer_name = "tiktoken:gpt-5"
except Exception:
    def count_tokens(text: str) -> int:
        return len(text.split())
    tokenizer_name = "fallback:whitespace_words"

rows = []

for category in CATEGORIES:
    folder = RESULTS_ROOT / category
    txt_files = sorted(folder.glob("*.txt"))

    token_counts = []
    char_counts = []
    word_counts = []

    for path in txt_files:
        text = path.read_text(encoding="utf-8", errors="replace").strip()
        token_counts.append(count_tokens(text))
        char_counts.append(len(text))
        word_counts.append(len(text.split()))

    def safe_mean(values):
        return stats.mean(values) if values else 0

    def safe_median(values):
        return stats.median(values) if values else 0

    def safe_stdev(values):
        return stats.stdev(values) if len(values) > 1 else 0

    rows.append({
        "category": category,
        "folder_exists": folder.exists(),
        "n_files": len(txt_files),
        "tokenizer": tokenizer_name,
        "tokens_total": sum(token_counts),
        "tokens_mean": round(safe_mean(token_counts), 2),
        "tokens_median": round(safe_median(token_counts), 2),
        "tokens_min": min(token_counts) if token_counts else 0,
        "tokens_max": max(token_counts) if token_counts else 0,
        "tokens_stdev": round(safe_stdev(token_counts), 2),
        "words_mean": round(safe_mean(word_counts), 2),
        "Words (min)": min(word_counts) if word_counts else 0,
        "Words (max)": max(word_counts) if word_counts else 0,
        "Words (SD)": round(stats.stdev(word_counts), 2) if len(word_counts) > 1 else 0,
        "chars_mean": round(safe_mean(char_counts), 2),
        "chars (min)": min(char_counts) if char_counts else 0,
        "chars (max)": max(char_counts) if char_counts else 0,
        "chars (SD)": round(stats.stdev(char_counts), 2) if len(char_counts) > 1 else 0,
    })

token_stats_df = pd.DataFrame(rows)
token_stats_df


,category,folder_exists,n_files,tokenizer,tokens_total,tokens_mean,tokens_median,tokens_min,tokens_max,tokens_stdev,words_mean,Words (min),Words (max),Words (SD),chars_mean,chars (min),chars (max),chars (SD)
0,nlg_brazilian_manager/e2e,True,24,tiktoken:gpt-5,127259,5302.46,5150.5,4218,6446,816.96,2435.88,2037,2996,281.22,15297.67,12752,18145,1700.27
1,nlg_brazilian_manager/default,True,24,tiktoken:gpt-5,136192,5674.67,5160.0,4411,9036,1297.92,2722.58,2157,4145,539.90,17089.71,13741,24842,3162.28
2,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,75036,5359.71,5377.5,4768,6054,427.74,2636.57,2339,2928,178.03,16869.21,15246,18670,1054.64
3,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,72610,5186.43,5066.0,4665,5787,358.73,2573.64,2326,2866,159.68,16369.86,14908,18419,1026.18
4,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,77174,5512.43,5413.5,4358,6394,591.59,2644.57,2147,2987,256.67,16925.50,13782,19126,1626.92
5,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,62451,4460.79,4425.5,3891,4853,288.84,2365.93,2152,2550,132.43,14932.21,13722,16052,773.93
6,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,66756,4768.29,4926.0,3054,6003,856.21,2436.07,1615,2972,417.32,15427.71,10122,19319,2673.75
7,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,68387,4884.79,4752.5,4334,5633,421.67,2402.21,2152,2773,179.37,15354.29,14048,18091,1238.60
8,nlg/final_report2025_us/gpt-5/workflow_False/o...,True,14,tiktoken:gpt-5,60892,4349.43,4445.5,3798,4707,287.47,2324.29,2158,2477,116.35,14769.79,13705,15721,708.07
9,nlg/final_report2025_us/gpt-5/workflow_False/o...,True,14,tiktoken:gpt-5,74600,5328.57,5372.0,4518,6532,516.37,2794.00,2378,3395,253.21,17610.07,15116,21982,1684.31


In [5]:
out_path = PROJECT_ROOT / "results" / "nlg_token_statistics_by_category.csv"
token_stats_df.to_csv(out_path, index=False)
out_path


PosixPath('/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/nlg_token_statistics_by_category.csv')

## LLM As Judge

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Optional
import json
import os
import re
import time

import pandas as pd
from IPython.display import JSON, display
from pydantic import BaseModel, ConfigDict, Field

try:
    from dotenv import load_dotenv
    load_dotenv(Path.home() / ".env")
    load_dotenv()
except Exception:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "llm-as-judge.ipynb").exists():
    fallback = Path("/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent")
    if fallback.exists():
        PROJECT_ROOT = fallback

BRAZILIAN_OUTPUT_VARIANTS = {
    "default": {
        "json": PROJECT_ROOT / "results" / "nlg_brazilian_manager" / "default" / "pt_br_manager_report_br_manager_2024-01-02.json",
        "txt": PROJECT_ROOT / "results" / "nlg_brazilian_manager" / "default" / "pt_br_manager_report_br_manager_2024-01-02.txt",
        "language": "pt-BR",
        "collection": "brazilian_manager",
    },
    "e2e": {
        "json": PROJECT_ROOT / "results" / "nlg_brazilian_manager" / "e2e" / "pt_br_manager_report_br_manager_2024-01-02.json",
        "txt": PROJECT_ROOT / "results" / "nlg_brazilian_manager" / "e2e" / "pt_br_manager_report_br_manager_2024-01-02.txt",
        "language": "pt-BR",
        "collection": "brazilian_manager",
    },
}

ENGLISH_OUTPUT_VARIANTS = {
    "default_reflection": {
        "json": PROJECT_ROOT / "results" / "nlg" / "final_report2025_us" / "gpt-5" / "workflow_True" / "openai" / "gpt-5" / "en" / "default" / "default_gpt-5_workflow_True_gpt-5_multi_stock_2025-01-31.json",
        "txt": PROJECT_ROOT / "results" / "nlg" / "final_report2025_us" / "gpt-5" / "workflow_True" / "openai" / "gpt-5" / "en" / "default" / "default_gpt-5_workflow_True_gpt-5_multi_stock_2025-01-31.txt",
        "language": "en",
        "collection": "us_stock_reflection",
    },
    "e2e_reflection": {
        "json": PROJECT_ROOT / "results" / "nlg" / "final_report2025_us" / "gpt-5" / "workflow_True" / "openai" / "gpt-5" / "en" / "e2e" / "e2e_gpt-5_workflow_True_gpt-5_multi_stock_2025-01-31.json",
        "txt": PROJECT_ROOT / "results" / "nlg" / "final_report2025_us" / "gpt-5" / "workflow_True" / "openai" / "gpt-5" / "en" / "e2e" / "e2e_gpt-5_workflow_True_gpt-5_multi_stock_2025-01-31.txt",
        "language": "en",
        "collection": "us_stock_reflection",
    },
}

OUTPUT_VARIANTS = {
    **{f"br_{name}": paths for name, paths in BRAZILIAN_OUTPUT_VARIANTS.items()},
    **{f"en_{name}": paths for name, paths in ENGLISH_OUTPUT_VARIANTS.items()},
}

RESULTS_DIR = PROJECT_ROOT / "results" / "validation" / "llm_judge_multi_model" / "br_en_manager"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
for variant_name, paths in OUTPUT_VARIANTS.items():
    label = paths["collection"].replace("_", " ").title()
    print(f"{label} output JSON ({variant_name}):", paths["json"])
    print(f"{label} output TXT ({variant_name}):", paths["txt"])
print("Results dir:", RESULTS_DIR)


## Configuration

GPT-5 is enabled when `OPENAI_API_KEY` is set. Set `AIXPLAIN_GEMINI_MODEL_ID` to the Gemini model asset ID available in your aixplain account. If you want Claude through aixplain instead of the direct Anthropic API, also set `AIXPLAIN_CLAUDE_MODEL_ID` and enable `claude_aixplain` below.


In [ ]:
# Judge configuration
OPENAI_JUDGE_MODEL = os.getenv("OPENAI_JUDGE_MODEL", "gpt-5")
CLAUDE_DIRECT_MODEL = os.getenv("CLAUDE_DIRECT_MODEL", "claude-3-7-sonnet-latest")
GEMINI_25_AIXPLAIN_MODEL_ID = os.getenv("GEMINI_25_AIXPLAIN_MODEL_ID", os.getenv("AIXPLAIN_GEMINI_MODEL_ID", "google/gemini-2.5-pro/google"))

JUDGE_MAX_OUTPUT_TOKENS = int(os.getenv("JUDGE_MAX_OUTPUT_TOKENS", "16000"))
JUDGE_MAX_RETRIES = int(os.getenv("JUDGE_MAX_RETRIES", "3"))
OVERWRITE = False

JUDGES = {
    "gpt5_openai": {
        "enabled": bool(os.getenv("OPENAI_API_KEY")),
        "provider": "openai",
        "model": OPENAI_JUDGE_MODEL,
        "folder": "GPT5_results",
    },
    "claude_3_7_sonnet": {
        "enabled": bool(os.getenv("ANTHROPIC_API_KEY")),
        "provider": "anthropic",
        "model": CLAUDE_DIRECT_MODEL,
        "folder": "Claude37_results",
    },
    "gemini_2_5_aixplain": {
        "enabled": bool((os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")) and GEMINI_25_AIXPLAIN_MODEL_ID),
        "provider": "aixplain",
        "model": GEMINI_25_AIXPLAIN_MODEL_ID,
        "folder": "Gemini25_results",
    },
}

# GPT-5 is also used for generating reduced/degraded variants.
PERTURBATION_MODEL = os.getenv("PERTURBATION_MODEL", "gpt-5")
PERTURBATION_LEVELS = [0.05, 0.10, 0.15]
RUN_PERTURBATIONS = True

pd.DataFrame([
    {"judge": name, **cfg} for name, cfg in JUDGES.items()
])


In [ ]:
JUDGE_INSTRUCTIONS_EN = """You are evaluating how well a Generated Report realises a given Input Bundle for a financial data-to-text task.

Your task:
1. Read the Input Bundle and the Generated Report carefully.
2. For each of the five Dimensions below, assign a score from 1 (lowest) to 5 (highest).
3. For each Dimension, give a short justification (one or two sentences).
4. Return only a single JSON object in the exact format specified. Do not include any extra text.

Dimensions:
No-Omissions: To what degree is ALL the information in the Input Bundle present in the Generated Report. Judge only against the structured data fields in the Input Bundle. Do not penalise the absence of year-over-year comparisons, prior-period figures, or growth rates that appear only in recommendation justification text and not in the structured data fields.
No-Additions: To what degree does the Generated Report include ONLY information from the Input Bundle. Report metadata supplied at the top of the Input Bundle is part of the input and must not be penalised as an addition.
Grammaticality: To what degree is the Generated Report grammatically correct in its target language.
Coherence: To what degree is the Generated Report logically ordered, internally consistent, and easy to follow.
Fluency: To what degree does the Generated Report read naturally and professionally in its target language.

Return this exact JSON shape:
{
  "No-Omissions": {"Justification": "", "Score": 1},
  "No-Additions": {"Justification": "", "Score": 1},
  "Grammaticality": {"Justification": "", "Score": 1},
  "Coherence": {"Justification": "", "Score": 1},
  "Fluency": {"Justification": "", "Score": 1}
}
"""

JUDGE_INSTRUCTIONS_PT_BR = """Você está avaliando quão bem um Relatório Gerado realiza um Pacote de Entrada em uma tarefa financeira de data-to-text.

Sua tarefa:
1. Leia cuidadosamente o Pacote de Entrada e o Relatório Gerado.
2. Para cada uma das cinco Dimensões abaixo, atribua uma nota de 1 (menor) a 5 (maior).
3. Para cada Dimensão, forneça uma justificativa curta (uma ou duas frases), em português brasileiro.
4. Retorne somente um único objeto JSON no formato exato especificado. Não inclua nenhum texto adicional.

Dimensões:
No-Omissions: Em que grau TODAS as informações do Pacote de Entrada estão presentes no Relatório Gerado. Julgue apenas com base nos campos de dados estruturados do Pacote de Entrada. Não penalize a ausência de comparações ano contra ano, números de períodos anteriores ou taxas de crescimento que apareçam apenas em justificativas de recomendação e não nos campos estruturados.
No-Additions: Em que grau o Relatório Gerado inclui SOMENTE informações presentes no Pacote de Entrada. Metadados do relatório fornecidos no início do Pacote de Entrada fazem parte da entrada e não devem ser penalizados como adição.
Grammaticality: Em que grau o Relatório Gerado está gramaticalmente correto em seu idioma-alvo.
Coherence: Em que grau o Relatório Gerado está logicamente ordenado, internamente consistente e fácil de acompanhar.
Fluency: Em que grau o Relatório Gerado soa natural e profissional em seu idioma-alvo.

Retorne este formato JSON exato. Mantenha os nomes das chaves em inglês para compatibilidade com o avaliador:
{
  "No-Omissions": {"Justification": "", "Score": 1},
  "No-Additions": {"Justification": "", "Score": 1},
  "Grammaticality": {"Justification": "", "Score": 1},
  "Coherence": {"Justification": "", "Score": 1},
  "Fluency": {"Justification": "", "Score": 1}
}
"""

JUDGE_INSTRUCTIONS_BY_LANGUAGE = {
    "en": JUDGE_INSTRUCTIONS_EN,
    "pt-br": JUDGE_INSTRUCTIONS_PT_BR,
    "pt_br": JUDGE_INSTRUCTIONS_PT_BR,
    "pt": JUDGE_INSTRUCTIONS_PT_BR,
}

# Backward-compatible default for older cells/helpers.
JUDGE_INSTRUCTIONS = JUDGE_INSTRUCTIONS_EN

DIMENSION_NAMES = ["No-Omissions", "No-Additions", "Grammaticality", "Coherence", "Fluency"]

class DimensionScore(BaseModel):
    Justification: str = Field(min_length=1)
    Score: int = Field(ge=1, le=5)

class JudgeScorecard(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    no_omissions: DimensionScore = Field(alias="No-Omissions")
    no_additions: DimensionScore = Field(alias="No-Additions")
    grammaticality: DimensionScore = Field(alias="Grammaticality")
    coherence: DimensionScore = Field(alias="Coherence")
    fluency: DimensionScore = Field(alias="Fluency")


In [ ]:
def safe_slug(text: str) -> str:
    value = (text or "sample").strip()
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)


def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        return json.loads(text[start:end + 1])


def flatten_scorecard(scorecard: JudgeScorecard) -> dict[str, Any]:
    payload = scorecard.model_dump(by_alias=True)
    row = {}
    for dimension in DIMENSION_NAMES:
        key = dimension.lower().replace("-", "_")
        row[f"{key}_score"] = payload[dimension]["Score"]
        row[f"{key}_justification"] = payload[dimension]["Justification"]
    return row


def judge_instructions_for_language(language: str) -> str:
    normalized = (language or "en").strip().lower().replace("_", "-")
    return JUDGE_INSTRUCTIONS_BY_LANGUAGE.get(normalized, JUDGE_INSTRUCTIONS_EN)


def make_judge_prompt(
    input_bundle: str,
    generated_report: str,
    previous_generated_report: str = "",
    language: str = "en",
) -> str:
    previous_section = previous_generated_report.strip() or "N/A"
    instructions = judge_instructions_for_language(language)
    if instructions == JUDGE_INSTRUCTIONS_PT_BR:
        return f"""{instructions}

Pacote de Entrada:
{input_bundle}

Relatório Gerado Anterior, se disponível:
{previous_section}

Relatório Gerado:
{generated_report}
"""
    return f"""{instructions}

Input Bundle:
{input_bundle}

Previous Generated Report, if available:
{previous_section}

Generated Report:
{generated_report}
"""


def extract_input_bundle(payload: dict[str, Any]) -> str:
    sample_metadata = payload.get("sample_metadata") if isinstance(payload.get("sample_metadata"), dict) else {}
    input_bundle = (
        payload.get("query")
        or payload.get("input")
        or payload.get("prompt")
        or payload.get("data_input")
        or payload.get("user_prompt")
        or sample_metadata.get("prompt_context")
        or sample_metadata.get("data_input")
    )
    if isinstance(input_bundle, (dict, list)):
        return json.dumps(input_bundle, ensure_ascii=False, indent=2)
    return str(input_bundle or "").strip()


def load_output_record(variant_name: str, paths: dict[str, Any]) -> dict[str, Any]:
    json_path = paths["json"]
    txt_path = paths["txt"]
    if not json_path.exists():
        raise FileNotFoundError(json_path)
    if not txt_path.exists():
        raise FileNotFoundError(txt_path)

    payload = json.loads(json_path.read_text(encoding="utf-8"))
    generated_text = payload.get("generated_text") or payload.get("final_response") or txt_path.read_text(encoding="utf-8")
    input_bundle = extract_input_bundle(payload)
    if not input_bundle:
        raise ValueError(f"Could not find input bundle/query/data_input in {json_path}")

    sample_metadata = payload.get("sample_metadata") if isinstance(payload.get("sample_metadata"), dict) else {}
    analysis_date = str(payload.get("analysis_date") or sample_metadata.get("analysis_date") or "unknown_date")
    sample_name = str(sample_metadata.get("sample_name") or json_path.stem)
    collection = paths.get("collection", "nlg")

    return {
        "sample_id": f"{collection}__{analysis_date}__{variant_name}",
        "sample_name": f"{sample_name}_{variant_name}",
        "workflow": variant_name,
        "collection": collection,
        "language": paths.get("language", ""),
        "analysis_date": analysis_date,
        "input_bundle": input_bundle.strip(),
        "generated_text": str(generated_text).strip(),
        "previous_generated_text": "",
        "source_json_path": str(json_path),
        "generated_txt_path": str(txt_path),
    }


records = [
    load_output_record(variant_name, paths)
    for variant_name, paths in OUTPUT_VARIANTS.items()
]
overview = pd.DataFrame([
    {k: v for k, v in record.items() if k not in {"input_bundle", "generated_text", "previous_generated_text"}}
    for record in records
])
display(overview)


## Judge Clients

In [ ]:
def call_anthropic(prompt: str, model: str) -> str:
    import requests

    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY is not set")

    response = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": model,
            "max_tokens": JUDGE_MAX_OUTPUT_TOKENS,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    return "\n".join(block.get("text", "") for block in payload.get("content", []) if block.get("type") == "text").strip()


def call_openai(prompt: str, model: str) -> str:
    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=JUDGE_MAX_OUTPUT_TOKENS,
    )
    return response.output_text.strip()


def extract_aixplain_output(result: Any) -> str:
    if hasattr(result, "data"):
        data = result.data
        if hasattr(data, "output"):
            return str(data.output).strip()
        if isinstance(data, dict):
            for key in ("output", "text", "data", "result"):
                if key in data:
                    return extract_aixplain_output(data[key])
    if isinstance(result, dict):
        for key in ("output", "text", "data", "result"):
            if key in result:
                return extract_aixplain_output(result[key])
    return str(result).strip()


def call_aixplain(prompt: str, model_id: str) -> str:
    if not model_id:
        raise RuntimeError("aiXplain model path is not configured")
    api_key = os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")
    if not api_key:
        raise RuntimeError("AIXPLAIN_API_KEY or TEAM_API_KEY is not set")

    from aixplain import Aixplain

    aix = Aixplain(api_key)
    model = aix.Model.get(model_id)
    try:
        result = model.run(text=prompt, temperature=0.0, max_tokens=JUDGE_MAX_OUTPUT_TOKENS)
    except TypeError:
        result = model.run(text=prompt)
    return extract_aixplain_output(result)


def call_unified_model(prompt: str, provider: str, model: str) -> str:
    import sys

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    from agents.llm_model import UnifiedModel, extract_text_output, resolve_model_config

    config = resolve_model_config(
        provider=provider,
        model_override=model,
        temperature=0.0,
        reasoning_effort=os.getenv("JUDGE_OPENAI_REASONING", "low") if provider == "openai" else None,
    )
    chain = UnifiedModel(provider=provider, **config).model_("You are a strict financial NLG judge. Return only the requested JSON object.")
    raw_output = chain.invoke({"input": prompt})
    return extract_text_output(raw_output)


def call_judge(prompt: str, judge_name: str, judge_config: dict[str, Any]) -> str:
    provider = judge_config["provider"]
    model = judge_config["model"]
    if provider == "openai":
        return call_openai(prompt, model)
    if provider == "anthropic":
        return call_anthropic(prompt, model)
    if provider == "aixplain":
        return call_aixplain(prompt, model)
    if provider == "groq":
        return call_unified_model(prompt, provider, model)
    raise ValueError(f"Unknown judge provider: {provider}")


In [ ]:
def judge_record(record: dict[str, Any], judge_name: str, judge_config: dict[str, Any]) -> dict[str, Any]:
    folder_name = judge_config.get("folder") or judge_name
    out_dir = RESULTS_DIR / folder_name / judge_name
    out_dir.mkdir(parents=True, exist_ok=True)
    output_path = out_dir / f"{safe_slug(record['sample_id'])}.json"

    if output_path.exists() and not OVERWRITE:
        payload = json.loads(output_path.read_text(encoding="utf-8"))
        scorecard = JudgeScorecard.model_validate(payload["scorecard"])
    else:
        prompt = make_judge_prompt(
            record["input_bundle"],
            record["generated_text"],
            record.get("previous_generated_text", ""),
            record.get("language", "en"),
        )
        last_error = None
        for attempt in range(1, JUDGE_MAX_RETRIES + 1):
            try:
                raw_text = call_judge(prompt, judge_name, judge_config)
                parsed = extract_json_object(raw_text)
                scorecard = JudgeScorecard.model_validate(parsed)
                payload = {
                    "sample_id": record["sample_id"],
                    "sample_name": record["sample_name"],
                    "judge": judge_name,
                    "judge_provider": judge_config["provider"],
                    "judge_model": judge_config["model"],
                    "prompt_language": record.get("language", "en"),
                    "source_json_path": record["source_json_path"],
                    "generated_txt_path": record["generated_txt_path"],
                    "scorecard": scorecard.model_dump(by_alias=True),
                    "raw_response": raw_text,
                }
                output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
                break
            except Exception as exc:
                last_error = exc
                if attempt == JUDGE_MAX_RETRIES:
                    raise
                time.sleep(2 * attempt)
        if last_error is not None and 'scorecard' not in locals():
            raise last_error

    row = {
        "sample_id": record["sample_id"],
        "sample_name": record["sample_name"],
        "judge": judge_name,
        "judge_provider": judge_config["provider"],
        "judge_model": judge_config["model"],
        "prompt_language": record.get("language", "en"),
        "judge_folder": judge_config.get("folder", judge_name),
        "score_path": str(output_path),
    }
    row.update(flatten_scorecard(scorecard))
    return row


def run_enabled_judges(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for judge_name, judge_config in JUDGES.items():
        if not judge_config.get("enabled"):
            print(f"Skipping {judge_name}: not enabled/configured")
            continue
        for record in records:
            print(f"Judging {record['sample_id']} with {judge_name}")
            rows.append(judge_record(record, judge_name, judge_config))
    return pd.DataFrame(rows)

judge_summary = run_enabled_judges(records)
if not judge_summary.empty:
    judge_summary.to_csv(RESULTS_DIR / "judge_summary.csv", index=False)
    display(judge_summary)
else:
    print("No judges are enabled. Set OPENAI_API_KEY, ANTHROPIC_API_KEY, and/or AIXPLAIN_GEMINI_MODEL_ID, then rerun.")


In [ ]:
SCORE_COLUMNS = [
    "no_omissions_score",
    "no_additions_score",
    "grammaticality_score",
    "coherence_score",
    "fluency_score",
]

if not judge_summary.empty:
    averaged_scores = (
        judge_summary
        .groupby(["sample_id", "sample_name"], as_index=False)[SCORE_COLUMNS]
        .mean()
    )
    averaged_scores["overall_mean_score"] = averaged_scores[SCORE_COLUMNS].mean(axis=1)
    averaged_scores.to_csv(RESULTS_DIR / "averaged_scores.csv", index=False)
    display(averaged_scores)


## Human Evaluation

This section prepares a blinded human-evaluation packet for the main NLG conditions and then analyses completed human ratings. The default design balances the Brazilian and US conditions by sampling 14 reports per condition, because the US reflection folders contain 14 reports while the Brazilian folders contain 24.

The API helper below uses the project wrapper in `agents/llm_model.py` for OpenAI, Anthropic, and aiXplain. Keep it optional: human ratings should come from the CSV completed by raters; model calls can be used for calibration, adjudication notes, or robustness checks without replacing human judgement.


In [ ]:
from __future__ import annotations

import hashlib
import random
from pathlib import Path
from typing import Any

import pandas as pd

HUMAN_EVAL_DIR = PROJECT_ROOT / "results" / "validation" / "human_eval_multi_model_robustness"
HUMAN_EVAL_DIR.mkdir(parents=True, exist_ok=True)

HUMAN_EVAL_SEED = 20260512
HUMAN_EVAL_N_PER_CONDITION = 14

HUMAN_EVAL_CONDITIONS = {
    "BR default": {
        "category": "nlg_brazilian_manager/default",
        "language": "pt-BR",
        "system_family": "Brazilian manager",
    },
    "BR e2e": {
        "category": "nlg_brazilian_manager/e2e",
        "language": "pt-BR",
        "system_family": "Brazilian manager",
    },
    "US default (reflection)": {
        "category": "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/default",
        "language": "en",
        "system_family": "US stocks",
    },
    "US e2e (reflection)": {
        "category": "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/e2e",
        "language": "en",
        "system_family": "US stocks",
    },
}

HUMAN_EVAL_SCORE_COLUMNS = [
    "no_omissions",
    "no_additions",
    "grammaticality",
    "coherence",
    "fluency",
    "overall_quality",
]

HUMAN_EVAL_RATING_COLUMNS = [
    "rater_id",
    "human_item_id",
    "no_omissions",
    "no_additions",
    "grammaticality",
    "coherence",
    "fluency",
    "overall_quality",
    "comment",
]


def stable_item_id(condition: str, path: Path) -> str:
    digest = hashlib.sha1(f"{condition}|{path.name}".encode("utf-8")).hexdigest()[:10]
    return f"HE-{digest}"


def extract_input_bundle_from_json(json_path: Path) -> str:
    if not json_path.exists():
        return ""
    try:
        payload = json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return ""

    sample_metadata = payload.get("sample_metadata") if isinstance(payload.get("sample_metadata"), dict) else {}
    input_bundle = (
        payload.get("query")
        or payload.get("input")
        or payload.get("prompt")
        or sample_metadata.get("prompt_context")
        or sample_metadata.get("data_input")
    )
    if isinstance(input_bundle, (dict, list)):
        return json.dumps(input_bundle, ensure_ascii=False, indent=2)
    return str(input_bundle or "").strip()


def collect_human_eval_candidates() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for condition, config in HUMAN_EVAL_CONDITIONS.items():
        folder = RESULTS_ROOT / config["category"]
        txt_files = sorted(path for path in folder.glob("*.txt") if path.is_file())
        for txt_path in txt_files:
            json_path = txt_path.with_suffix(".json")
            generated_text = txt_path.read_text(encoding="utf-8", errors="replace").strip()
            rows.append({
                "human_item_id": stable_item_id(condition, txt_path),
                "condition": condition,
                "category": config["category"],
                "language": config["language"],
                "system_family": config["system_family"],
                "report_name": txt_path.stem,
                "report_path": str(txt_path.relative_to(PROJECT_ROOT)),
                "json_path": str(json_path.relative_to(PROJECT_ROOT)) if json_path.exists() else "",
                "input_bundle": extract_input_bundle_from_json(json_path),
                "generated_report": generated_text,
                "word_count": len(generated_text.split()),
            })
    return pd.DataFrame(rows)


candidate_reports = collect_human_eval_candidates()
condition_counts = candidate_reports.groupby("condition").size().rename("available_reports").reset_index()
display(condition_counts)


In [ ]:
def build_balanced_human_eval_packet(candidates: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = random.Random(HUMAN_EVAL_SEED)
    sampled_frames = []
    for condition in HUMAN_EVAL_CONDITIONS:
        condition_rows = candidates[candidates["condition"] == condition].copy()
        if condition_rows.empty:
            print(f"No reports found for condition: {condition}")
            continue
        records = condition_rows.to_dict("records")
        rng.shuffle(records)
        sampled_frames.append(pd.DataFrame(records[:HUMAN_EVAL_N_PER_CONDITION]))

    if not sampled_frames:
        return pd.DataFrame(), pd.DataFrame()

    packet = pd.concat(sampled_frames, ignore_index=True)
    packet_records = packet.to_dict("records")
    rng.shuffle(packet_records)
    packet = pd.DataFrame(packet_records).reset_index(drop=True)
    packet.insert(0, "presentation_order", range(1, len(packet) + 1))

    public_packet = packet[[
        "presentation_order",
        "human_item_id",
        "language",
        "system_family",
        "report_name",
        "input_bundle",
        "generated_report",
    ]].copy()

    rating_sheet = public_packet[[
        "presentation_order",
        "human_item_id",
        "language",
        "system_family",
        "report_name",
    ]].copy()
    for column in HUMAN_EVAL_RATING_COLUMNS:
        if column not in rating_sheet.columns:
            rating_sheet[column] = ""

    key = packet[[
        "presentation_order",
        "human_item_id",
        "condition",
        "category",
        "report_path",
        "json_path",
        "word_count",
    ]].copy()
    return public_packet, key


human_packet, human_key = build_balanced_human_eval_packet(candidate_reports)

packet_jsonl = HUMAN_EVAL_DIR / "human_eval_packets.jsonl"
packet_csv = HUMAN_EVAL_DIR / "human_eval_packets.csv"
rating_sheet_csv = HUMAN_EVAL_DIR / "human_eval_rating_sheet.csv"
key_csv = HUMAN_EVAL_DIR / "human_eval_key_private.csv"

if not human_packet.empty:
    human_packet.to_json(packet_jsonl, orient="records", lines=True, force_ascii=False)
    human_packet.to_csv(packet_csv, index=False)
    human_key.to_csv(key_csv, index=False)

    rating_sheet = human_packet[["presentation_order", "human_item_id", "language", "system_family", "report_name"]].copy()
    for column in HUMAN_EVAL_RATING_COLUMNS:
        if column not in rating_sheet.columns:
            rating_sheet[column] = ""
    rating_sheet = rating_sheet[HUMAN_EVAL_RATING_COLUMNS + ["presentation_order", "language", "system_family", "report_name"]]
    rating_sheet.to_csv(rating_sheet_csv, index=False)

    print("Human evaluation files written:")
    print("-", packet_jsonl.relative_to(PROJECT_ROOT))
    print("-", packet_csv.relative_to(PROJECT_ROOT))
    print("-", rating_sheet_csv.relative_to(PROJECT_ROOT))
    print("-", key_csv.relative_to(PROJECT_ROOT), "(private key; do not share with raters)")
    display(human_key.groupby("condition").size().rename("n").reset_index())
else:
    print("No human evaluation packet was created because no candidate reports were found.")


In [ ]:
# Optional: provider calls for calibration/adjudication notes using agents/llm_model.py.
# This uses the same project wrapper for OpenAI, Anthropic, and aiXplain.

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.llm_model import UnifiedModel, extract_text_output, resolve_model_config

HUMAN_EVAL_ASSISTANT_SYSTEM_PROMPT = """You are assisting a financial NLG human-evaluation study.
Use the rubric exactly as written. Do not replace human ratings.
Return concise, actionable calibration or adjudication notes only.
"""

HUMAN_EVAL_API_MODELS = {
    "openai": {
        "enabled": bool(os.getenv("OPENAI_API_KEY")),
        "provider": "openai",
        "model": os.getenv("HUMAN_EVAL_OPENAI_MODEL", "gpt-5-mini"),
        "reasoning_effort": os.getenv("HUMAN_EVAL_OPENAI_REASONING", "low"),
    },
    "anthropic": {
        "enabled": bool(os.getenv("ANTHROPIC_API_KEY")),
        "provider": "anthropic",
        "model": os.getenv("HUMAN_EVAL_ANTHROPIC_MODEL", "claude-sonnet-4-5"),
        "reasoning_effort": None,
    },
    "aixplain": {
        "enabled": bool((os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")) and os.getenv("HUMAN_EVAL_AIXPLAIN_MODEL_ID", os.getenv("AIXPLAIN_GEMINI_MODEL_ID", "google/gemini-2.5-pro/google"))),
        "provider": "aixplain",
        "model": os.getenv("HUMAN_EVAL_AIXPLAIN_MODEL_ID", os.getenv("AIXPLAIN_GEMINI_MODEL_ID", "google/gemini-2.5-pro/google")),
        "reasoning_effort": None,
    },
}


def call_human_eval_model(provider_name: str, prompt: str) -> str:
    config = HUMAN_EVAL_API_MODELS[provider_name]
    if not config.get("enabled"):
        raise RuntimeError(f"{provider_name} is not enabled. Check API key and model environment variables.")

    model_kwargs = resolve_model_config(
        provider=config["provider"],
        model_override=config["model"],
        temperature=0.0,
        reasoning_effort=config.get("reasoning_effort"),
    )
    chain = UnifiedModel(provider=config["provider"], **model_kwargs).model_(HUMAN_EVAL_ASSISTANT_SYSTEM_PROMPT)
    raw_output = chain.invoke({"input": prompt})
    return extract_text_output(raw_output)


def make_calibration_prompt(example_row: pd.Series) -> str:
    return f"""Review this human-evaluation item for rubric calibration.

Scores are 1 to 5 for: no_omissions, no_additions, grammaticality, coherence, fluency, overall_quality.
Flag any likely ambiguity a human rater should be aware of. Do not assign final scores.

Language: {example_row.get('language', '')}
System family: {example_row.get('system_family', '')}
Input bundle:
{example_row.get('input_bundle', '')[:6000]}

Generated report:
{example_row.get('generated_report', '')[:6000]}
"""

# Example usage, intentionally disabled by default to avoid API cost:
# if not human_packet.empty:
#     example_prompt = make_calibration_prompt(human_packet.iloc[0])
#     print(call_human_eval_model("openai", example_prompt))

pd.DataFrame([
    {"name": name, **{k: v for k, v in config.items() if k != "reasoning_effort"}}
    for name, config in HUMAN_EVAL_API_MODELS.items()
])


In [ ]:
HUMAN_COMPLETED_RATINGS_CSV = HUMAN_EVAL_DIR / "human_eval_completed.csv"


def load_and_summarise_human_ratings(completed_csv: Path = HUMAN_COMPLETED_RATINGS_CSV) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not completed_csv.exists():
        print(f"Completed human ratings file not found yet: {completed_csv.relative_to(PROJECT_ROOT)}")
        print("Fill human_eval_rating_sheet.csv, save it as human_eval_completed.csv in the same folder, then rerun this cell.")
        return pd.DataFrame(), pd.DataFrame()

    ratings = pd.read_csv(completed_csv)
    missing = [column for column in HUMAN_EVAL_RATING_COLUMNS if column not in ratings.columns]
    if missing:
        raise ValueError(f"Missing required rating columns: {missing}")

    for column in HUMAN_EVAL_SCORE_COLUMNS:
        ratings[column] = pd.to_numeric(ratings[column], errors="coerce")

    key = pd.read_csv(HUMAN_EVAL_DIR / "human_eval_key_private.csv")
    merged = ratings.merge(key, on="human_item_id", how="left", validate="many_to_one")
    if merged["condition"].isna().any():
        missing_ids = merged.loc[merged["condition"].isna(), "human_item_id"].dropna().unique().tolist()
        raise ValueError(f"Some human_item_id values are not in the private key: {missing_ids[:10]}")

    summary = (
        merged
        .groupby("condition", as_index=False)
        .agg(
            n_items=("human_item_id", "nunique"),
            n_ratings=("human_item_id", "size"),
            **{f"{column}_mean": (column, "mean") for column in HUMAN_EVAL_SCORE_COLUMNS},
            **{f"{column}_sd": (column, "std") for column in HUMAN_EVAL_SCORE_COLUMNS},
        )
    )
    summary["overall_rank"] = summary["overall_quality_mean"].rank(ascending=False, method="min")

    merged.to_csv(HUMAN_EVAL_DIR / "human_eval_ratings_with_conditions.csv", index=False)
    summary.to_csv(HUMAN_EVAL_DIR / "human_eval_summary_by_condition.csv", index=False)
    return merged, summary


def pairwise_rater_correlations(merged_ratings: pd.DataFrame, score_column: str = "overall_quality") -> pd.DataFrame:
    if merged_ratings.empty or "rater_id" not in merged_ratings.columns:
        return pd.DataFrame()

    wide = merged_ratings.pivot_table(index="human_item_id", columns="rater_id", values=score_column, aggfunc="mean")
    rows = []
    raters = list(wide.columns)
    for i, left in enumerate(raters):
        for right in raters[i + 1:]:
            paired = wide[[left, right]].dropna()
            rows.append({
                "score_column": score_column,
                "rater_a": left,
                "rater_b": right,
                "n_overlap": len(paired),
                "pearson": paired[left].corr(paired[right], method="pearson") if len(paired) > 1 else pd.NA,
                "spearman": paired[left].corr(paired[right], method="spearman") if len(paired) > 1 else pd.NA,
            })
    return pd.DataFrame(rows)


human_ratings, human_summary = load_and_summarise_human_ratings()
if not human_summary.empty:
    display(human_summary)
    display(pairwise_rater_correlations(human_ratings, "overall_quality"))


## GPT-5 Perturbation Experiment

For each perturbation level, GPT-5 creates two degraded variants:

- `input_reduced`: reduce the input bundle while keeping the generated output unchanged.
- `output_reduced`: reduce the generated output while keeping the input bundle unchanged.

The same enabled judges then score the original-vs-perturbed pairs. A capable judge should generally reduce omission/addition-related scores when important information is removed or degraded.


In [ ]:
def call_openai_text(prompt: str, model: str = PERTURBATION_MODEL) -> str:
    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=12000,
    )
    return response.output_text.strip()


def make_reduction_prompt(text: str, side: str, level: float, language: str) -> str:
    percent = int(round(level * 100))
    return f"""You are preparing a controlled robustness test for an LLM-as-judge experiment.

Task: reduce the informational completeness/quality of the {side} by approximately {percent}% while preserving the same language ({language}) and the same general format.

Rules:
- Remove or weaken substantive facts, figures, constraints, or details.
- Do not add new facts.
- Do not explain what you changed.
- Return only the reduced text.

Text:
{text}
"""


def build_perturbed_records(base_records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    perturbed = []
    for record in base_records:
        for level in PERTURBATION_LEVELS:
            input_reduced = call_openai_text(make_reduction_prompt(record["input_bundle"], "input bundle", level, record["language"]))
            rec = dict(record)
            rec["sample_id"] = f"{record['sample_id']}__input_reduced_{level:.2f}"
            rec["sample_name"] = f"{record['sample_name']} input reduced {level:.2f}"
            rec["input_bundle"] = input_reduced
            rec["perturbation_type"] = "input_reduced"
            rec["perturbation_level"] = level
            perturbed.append(rec)

            output_reduced = call_openai_text(make_reduction_prompt(record["generated_text"], "generated report", level, record["language"]))
            rec = dict(record)
            rec["sample_id"] = f"{record['sample_id']}__output_reduced_{level:.2f}"
            rec["sample_name"] = f"{record['sample_name']} output reduced {level:.2f}"
            rec["generated_text"] = output_reduced
            rec["perturbation_type"] = "output_reduced"
            rec["perturbation_level"] = level
            perturbed.append(rec)
    return perturbed

perturbed_records_path = RESULTS_DIR / "perturbed_records.json"
if RUN_PERTURBATIONS:
    if perturbed_records_path.exists() and not OVERWRITE:
        perturbed_records = json.loads(perturbed_records_path.read_text(encoding="utf-8"))
    else:
        perturbed_records = build_perturbed_records(records)
        perturbed_records_path.write_text(json.dumps(perturbed_records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Perturbed records: {len(perturbed_records)}")
else:
    perturbed_records = []
    print("Perturbations disabled")


In [ ]:
if perturbed_records:
    perturbation_summary = run_enabled_judges(perturbed_records)
    if not perturbation_summary.empty:
        meta = pd.DataFrame([
            {
                "sample_id": record["sample_id"],
                "perturbation_type": record.get("perturbation_type"),
                "perturbation_level": record.get("perturbation_level"),
            }
            for record in perturbed_records
        ])
        perturbation_summary = perturbation_summary.merge(meta, on="sample_id", how="left")
        perturbation_summary.to_csv(RESULTS_DIR / "perturbation_judge_summary.csv", index=False)
        display(perturbation_summary)
else:
    perturbation_summary = pd.DataFrame()


In [ ]:
if not perturbation_summary.empty:
    perturbation_average = (
        perturbation_summary
        .groupby(["sample_id", "sample_name", "perturbation_type", "perturbation_level"], as_index=False)[SCORE_COLUMNS]
        .mean()
    )
    perturbation_average["overall_mean_score"] = perturbation_average[SCORE_COLUMNS].mean(axis=1)
    perturbation_average.to_csv(RESULTS_DIR / "perturbation_averaged_scores.csv", index=False)

    baseline_average = averaged_scores.assign(perturbation_type="baseline", perturbation_level=0.0) if 'averaged_scores' in globals() and not averaged_scores.empty else pd.DataFrame()
    robustness_table = pd.concat([baseline_average, perturbation_average], ignore_index=True, sort=False)
    robustness_table.to_csv(RESULTS_DIR / "robustness_table.csv", index=False)
    display(robustness_table)


## Optional: Find aiXplain Model IDs

Run this helper if you need to discover the exact Gemini or Claude model asset IDs available to your aixplain account. The returned IDs can be copied into `AIXPLAIN_GEMINI_MODEL_ID` or `AIXPLAIN_CLAUDE_MODEL_ID` in your environment.


In [ ]:
def get_aixplain_model(model_path: str = "google/gemini-2.5-pro/google"):
    api_key = os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")
    if not api_key:
        raise RuntimeError("AIXPLAIN_API_KEY or TEAM_API_KEY is not set")
    from aixplain import Aixplain

    aix = Aixplain(api_key)
    return aix.Model.get(model_path)

# Example:
# gemini_25 = get_aixplain_model("google/gemini-2.5-pro/google")
# result = gemini_25.run(text="Test prompt")
# print(result)
